# Topic 1 – HTTP & REST in Practice 🎮
### Learning APIs with game data (PokéAPI, D&D 5e API, httpbin)

**Goal:** see every concept from the *HTTP & REST Fundamentals* slides working in real code.

| Service | Why we use it | Auth |
|---|---|---|
| [PokéAPI](https://pokeapi.co) | Game characters & stats (GET, query params, pagination, 404) | None |
| [D&D 5e API](https://www.dnd5eapi.co) | RPG monsters (clean resource design) | None |
| [httpbin.org](https://httpbin.org) | A test server that echoes what we send (POST, headers) | None |

All are free, public and need no API key. Be polite: don't hammer them with loops of thousands of requests.

**Roadmap**
1. First GET request
2. Anatomy of the response (status, headers, body)
3. JSON → Python objects
4. Query parameters & pagination
5. Status codes & errors (404)
6. Resource design with the D&D API
7. POST with a JSON body (httpbin)
8. Mini project: a battle between two characters
9. Bridge to our ML API (`POST /predict`)

## Step -1 – Install dependencies (run once)
This installs `requests` into **the kernel you selected**, so it works in any environment (venv, system, Cursor, VS Code, Jupyter).

In [ ]:
%pip install -q requests

## Step 0 – Setup
`requests` is the standard Python library for HTTP calls.

In [1]:
import requests
import json

print("requests version:", requests.__version__)

requests version: 2.32.3


## Step 1 – Our first GET request
**GET** means *"give me this resource"*. The URL identifies the resource: here, the Pokémon called `pikachu`.

In [2]:
url = "https://pokeapi.co/api/v2/pokemon/pikachu"
response = requests.get(url, timeout=10)

print(response)                 # <Response [200]>
print("Status code:", response.status_code)

<Response [200]>
Status code: 200


## Step 2 – Anatomy of the response
From the slides: a response has **status code**, **headers** and a **body**.

In [3]:
print("STATUS :", response.status_code, response.reason)
print("URL    :", response.url)
print("ELAPSED:", response.elapsed.total_seconds(), "seconds")
print()
print("HEADERS (selected):")
for key in ["Content-Type", "Content-Length", "Cache-Control", "Server"]:
    print(f"  {key}: {response.headers.get(key)}")

STATUS : 200 OK
URL    : https://pokeapi.co/api/v2/pokemon/pikachu
ELAPSED: 1.480407 seconds

HEADERS (selected):
  Content-Type: application/json; charset=utf-8
  Content-Length: None
  Cache-Control: public, max-age=86400, s-maxage=86400
  Server: cloudflare


In [4]:
# The body is text; JSON is just text that follows a strict format
print(response.text[:300], "...")

{"id":25,"name":"pikachu","base_experience":112,"height":4,"is_default":true,"order":35,"weight":60,"abilities":[{"is_hidden":false,"slot":1,"ability":{"name":"static","url":"https://pokeapi.co/api/v2/ability/9/"}},{"is_hidden":true,"slot":3,"ability":{"name":"lightning-rod","url":"https://pokeapi.c ...


## Step 3 – JSON → Python
`response.json()` converts the JSON body into Python dicts and lists. Now we can use it like any data.

In [5]:
data = response.json()

print("Type:", type(data))
print("Top-level keys:", list(data.keys())[:10], "...")
print()
print("Name  :", data["name"])
print("Height:", data["height"])
print("Weight:", data["weight"])
print("Types :", [t["type"]["name"] for t in data["types"]])

Type: <class 'dict'>
Top-level keys: ['id', 'name', 'base_experience', 'height', 'is_default', 'order', 'weight', 'abilities', 'past_abilities', 'forms'] ...

Name  : pikachu
Height: 4
Weight: 60
Types : ['electric']


In [6]:
# Turn the raw stats into a game "character card"
stats = {s["stat"]["name"]: s["base_stat"] for s in data["stats"]}
print(f"=== {data['name'].upper()} ===")
for stat, value in stats.items():
    print(f"{stat:<18} {'#' * (value // 10)} {value}")

=== PIKACHU ===
hp                 ### 35
attack             ##### 55
defense            #### 40
special-attack     ##### 50
special-defense    ##### 50
speed              ######### 90


## Step 4 – Query parameters & pagination
In the URL, the **path** picks the resource and the **query parameters** (`?limit=5&offset=0`) tweak the result.
A list endpoint should never return everything: it returns a *page*.

In [7]:
url = "https://pokeapi.co/api/v2/pokemon"
params = {"limit": 5, "offset": 0}

page1 = requests.get(url, params=params, timeout=10)
print("Final URL sent:", page1.url)

body = page1.json()
print("Total available:", body["count"])
print("Next page URL  :", body["next"])
print("Items on page 1:", [p["name"] for p in body["results"]])

Final URL sent: https://pokeapi.co/api/v2/pokemon?limit=5&offset=0
Total available: 1351
Next page URL  : https://pokeapi.co/api/v2/pokemon?offset=5&limit=5
Items on page 1: ['bulbasaur', 'ivysaur', 'venusaur', 'charmander', 'charmeleon']


In [8]:
# Page 2: only the offset changes
params["offset"] = 5
page2 = requests.get(url, params=params, timeout=10).json()
print("Items on page 2:", [p["name"] for p in page2["results"]])

Items on page 2: ['charizard', 'squirtle', 'wartortle', 'blastoise', 'caterpie']


## Step 5 – Status codes & errors
A good client **checks the status code** instead of assuming success. Let's ask for a Pokémon that doesn't exist.

In [9]:
bad = requests.get("https://pokeapi.co/api/v2/pokemon/not-a-real-pokemon", timeout=10)
print("Status:", bad.status_code, bad.reason)
print("Body  :", bad.text[:100])

Status: 404 Not Found
Body  : {"status":404,"message":"Not Found"}


In [10]:
def get_json(url, **kwargs):
    """Safe GET: timeout + status check + clear error messages."""
    try:
        r = requests.get(url, timeout=10, **kwargs)
        r.raise_for_status()            # raises for any 4xx / 5xx
        return r.json()
    except requests.exceptions.HTTPError as e:
        print(f"HTTP error {e.response.status_code} for {url}")
    except requests.exceptions.Timeout:
        print("The server took too long (timeout)")
    except requests.exceptions.ConnectionError:
        print("Could not connect (check internet / URL)")
    return None


print(get_json("https://pokeapi.co/api/v2/pokemon/charmander")["name"])
print(get_json("https://pokeapi.co/api/v2/pokemon/xyz"))

charmander


HTTP error 404 for https://pokeapi.co/api/v2/pokemon/xyz
None


## Step 6 – Resource design: the D&D 5e API
Notice the pattern: **plural nouns** and predictable URLs → `/api/monsters`, `/api/monsters/goblin`.
This is the style we studied in *Topic 2 – API Design*.

In [11]:
BASE = "https://www.dnd5eapi.co"

root = get_json(f"{BASE}/api")
print("Available resources:")
for name, path in list(root.items())[:8]:
    print(f"  {name:<18} -> {path}")

Available resources:
  ability-scores     -> /api/2014/ability-scores
  alignments         -> /api/2014/alignments
  backgrounds        -> /api/2014/backgrounds
  classes            -> /api/2014/classes
  conditions         -> /api/2014/conditions
  damage-types       -> /api/2014/damage-types
  equipment          -> /api/2014/equipment
  equipment-categories -> /api/2014/equipment-categories


In [12]:
goblin = get_json(f"{BASE}/api/monsters/goblin")
print("Monster :", goblin["name"])
print("HP      :", goblin["hit_points"])
print("Armor   :", goblin["armor_class"][0]["value"])
print("Actions :", [a["name"] for a in goblin["actions"]])

Monster : Goblin
HP      : 7
Armor   : 15
Actions : ['Scimitar', 'Shortbow']


In [13]:
# Filtering with a query parameter
monsters = get_json(f"{BASE}/api/monsters", params={"name": "dragon"})
print("Monsters with 'dragon':", monsters["count"])
print([m["name"] for m in monsters["results"][:5]])

Monsters with 'dragon': 43
['Adult Black Dragon', 'Adult Blue Dragon', 'Adult Brass Dragon', 'Adult Bronze Dragon', 'Adult Copper Dragon']


## Step 7 – POST: sending data with a JSON body
**GET** reads. **POST** sends data (just like our future `POST /predict`).
`httpbin.org` is a *mirror*: it answers with exactly what it received, so we can inspect our request.

In [14]:
player = {"player": "Luis", "level": 7, "score": 4820}

r = requests.post("https://httpbin.org/post", json=player, timeout=10)

print("Status:", r.status_code)
echo = r.json()
print("Server received JSON   :", echo["json"])
print("Content-Type we sent   :", echo["headers"]["Content-Type"])

Status: 200
Server received JSON   : {'level': 7, 'player': 'Luis', 'score': 4820}
Content-Type we sent   : application/json


In [ ]:
# Custom headers (this is how API keys / tokens travel later)
headers = {"X-Game-Token": "demo-123", "Accept": "application/json"}
r = requests.get("https://httpbin.org/headers", headers=headers, timeout=10)
print(json.dumps(r.json(), indent=2))

In [16]:
# httpbin can also fabricate any status code, handy for testing error handling
for status in [200, 201, 404, 500]:
    r = requests.get(f"https://httpbin.org/status/{status}", timeout=10)
    print(status, "->", r.status_code, r.reason)

200 -> 200 OK


201 -> 201 CREATED


404 -> 404 NOT FOUND


500 -> 500 INTERNAL SERVER ERROR


## Step 8 – Mini project: Battle of two characters
We combine everything: GET, JSON parsing, helper function and error handling.
The winner is the one with the higher total of base stats.

In [17]:
def load_fighter(name):
    data = get_json(f"https://pokeapi.co/api/v2/pokemon/{name.lower()}")
    if data is None:
        return None
    stats = {s["stat"]["name"]: s["base_stat"] for s in data["stats"]}
    return {"name": data["name"], "stats": stats, "power": sum(stats.values())}


def battle(a, b):
    f1, f2 = load_fighter(a), load_fighter(b)
    if not f1 or not f2:
        print("Battle cancelled: one fighter was not found.")
        return
    print(f"{f1['name'].upper()} ({f1['power']})  VS  {f2['name'].upper()} ({f2['power']})")
    if f1["power"] == f2["power"]:
        print("It's a draw!")
    else:
        winner = f1 if f1["power"] > f2["power"] else f2
        print("Winner:", winner["name"].upper())


battle("pikachu", "bulbasaur")
battle("mewtwo", "magikarp")
battle("pikachu", "fake-monster")

PIKACHU (320)  VS  BULBASAUR (318)
Winner: PIKACHU


MEWTWO (680)  VS  MAGIKARP (200)
Winner: MEWTWO


HTTP error 404 for https://pokeapi.co/api/v2/pokemon/fake-monster
Battle cancelled: one fighter was not found.


## Step 9 – Bridge to our ML project
Look at what we practiced and where it goes next:

| Today (consuming) | Next topics (building) |
|---|---|
| `requests.get(pokeapi...)` | We build our own server with **FastAPI** |
| `requests.post(httpbin, json=player)` | `POST /predict` with model features |
| `response.json()` | Pydantic schemas validate input/output |
| status codes, timeouts | Same error rules in our API |

**Idea:** instead of predicting house prices, we could predict **a character's power** from its stats using linear regression, a game-flavored ML project that reuses everything above.

### Exercises
1. Show the top 10 heaviest Pokémon on the first 20 results (use `limit=20`).
2. Add a `type` filter: use `https://pokeapi.co/api/v2/type/fire` and list its Pokémon.
3. Make `battle()` compare only `attack + speed`.
4. Use `httpbin.org/delay/5` with `timeout=2` and handle the `Timeout`.